In [ ]:
import pandas as pd
import numpy as np
from coffea import util
import itertools
import os, sys
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
import hist
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
import warnings
hep.style.use("CMS")

sys.path.append('../python/')
from functions import loadCoffeaFile, getLabelMap, getCoffeaFilenames, plotBackgroundEstimate, getHist


In [ ]:
def getHist(hname, ds, bkgest, year, sum_axes=[], integrate_axes={}, masspoint=''): # Copied and modified from `closureTest.ipynb`
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames(False, useOldHTcut)
    
    cfiles = []
    bkgest_str = np.where([bkgest], 'weighted', 'unweighted')[0]
    # print(coffeaFiles[ds][bkgest_str][year])
    
     
    for key, file in coffeaFiles[ds][bkgest_str][year].items():
        if masspoint != '':
            # print('Grabbing signal...')
            if masspoint in key:
                loaded_file = LoadedFiles[ds][bkgest_str][year][masspoint]
                sum_axes_dict = {ax:sum for ax in sum_axes}
                histo = loaded_file[hname][integrate_axes][sum_axes_dict]
                histo = histo * (lumi[year] * 1.0 / loaded_file['cutflow']['sumw'])
                return histo

        loaded_file = LoadedFiles[ds][bkgest_str][year][key]
        cfiles.append(loaded_file)
        
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
    
    hists = []
    
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])   
    
#     # sum all hists from dataset eras or pt bins
    
    histo = hists[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]
            
    return histo

In [ ]:
useOldHTcut = False

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530./10.,
    "2018": 59800./10., #59740./10., #Blinding
    "Full": 46053. # 137190. Blinded
}

systematics = [
        'jes',
        'jer',
        'pileup',
        'pdf',
        'q2',
        'btag',
        'toptagsf',
        'toptagxs',
        'lumi',
        'prefiring'
    ]

In [ ]:
# load histograms and get scale factors
coffeaFiles = getCoffeaFilenames(False, useOldHTcut, True)

LoadedFiles = {
    'RSGluon':{
        'unweighted':{
            '2016APV': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2016': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2017': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None}
        }
    },
    'ZPrime1':{
        'unweighted':{
            '2016APV': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None},
            '2016': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None},
            '2017': {'1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '3000':None, '3500':None, '4000':None, '4500':None}
        }
    },
    'ZPrime10':{
        'unweighted':{
            '2016APV': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2016': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2017': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None}
        }
    },
    'ZPrime30':{
        'unweighted':{
            '2016APV': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2016': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2017': {'1000':None, '1200':None, '1400':None, '1600':None, '1800':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None}
        }
    },
    'ZPrimeDM':{
        'unweighted':{
            '2016APV': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2016': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None},
            '2017': {'1000':None, '1500':None, '2000':None, '2500':None, '3000':None, '3500':None, '4000':None, '4500':None, '5000':None}
        }
    }
}

In [ ]:
bkgest_str = 'unweighted'
for year in ['2016', '2017']:
    for ds in ['RSGluon']: #, 'ZPrime1', 'ZPrime10', 'ZPrime30', 'ZPrimeDM']:
        for key, file in coffeaFiles[ds][bkgest_str][year].items():
            LoadedFiles[ds][bkgest_str][year][key] = (util.load(file))
            print(file + ' loaded') # for masspoint ' + key)

In [ ]:
signal_cats = [ i for label, i in label_to_int_dict.items() if '2t' in label]
category = '0bcen'
signal_cats = label_to_int_dict['2t'+category]

hist1 = getHist('ttbarmass', 'RSGluon', False, '2016', [], {'anacat':signal_cats, 'systematic':'nominal'}, '1000')
hist1
# np.sum(hist1.values())

In [ ]:
# analysis categories #
label_dict = LoadedFiles['RSGluon']['unweighted']['2016']['1000']['analysisCategories']
label_to_int_dict = {label: i for i, label in label_dict.items()}
print(label_dict)

LoadedFiles['RSGluon']['unweighted']['2016']['1000']['ttbarmass'][{'anacat':signal_cats, 'systematic':'nominal'}]

In [ ]:
categories = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']
for cat in categories:
    signal_cats = label_to_int_dict['2t'+cat]
    hist1 = getHist('ttbarmass', 'RSGluon', False, '2016', [], {'anacat':signal_cats, 'systematic':'nominal'}, '4000')
    # hist2 = getHist('ttbarmass', 'RSGluon', False, '2016APV', [], {'anacat':signal_cats, 'systematic':'nominal'}, '4000')
    
    #hist = hist2 + hist1
    hist = hist1
    print(f'{cat}:\t\t{np.sum(hist.values())}\n')

In [ ]:
warnings.filterwarnings("ignore")

# Color = ['red', 'pink', 'orange', 'green', 'xkcd:pale gold', 'cyan', 'blue', 'violet', 'grey']


Color = ['red', 'green', 'blue', 'violet']
mass = ['1000', '2000', '3000', '4000']
xaxis = 'ttbarmass'

year1 = '2016'
year2 = '2017'
xaxis = 'ttbarmass'

for cat in categories:
    signal_cats = label_to_int_dict['2t'+cat]
    # fig = plt.figure(icat)
    fig, (ax, bx) = plt.subplots(
    ncols = 2,
    figsize=(18,6),
    sharex=True
)
    Text = f''
    hep.cms.label('       '+Text, ax=ax, data=False, lumi='{0:0.1f}'.format(lumi[year1]/1000.), year=year1, loc=0, fontsize=20)   
    hep.cms.label('       '+Text, ax=bx, data=False, lumi='{0:0.1f}'.format(lumi[year2]/1000.), year=year2, loc=0, fontsize=20)   
    yMax1, yMax2 = [], []
    for i in range(4): 
        RSGluonhists1 = getHist(xaxis, 'RSGluon', False, year1, [], {'anacat':signal_cats, 'systematic':'nominal'}, mass[i])
        hep.histplot(RSGluonhists1, ax=ax, color=Color[i], lw=3, label='Mass ['+mass[i]+' GeV]', yerr=False)
        yMax1.append(np.max(RSGluonhists1.values()))
        
        RSGluonhists2 = getHist(xaxis, 'RSGluon', False, year2, [], {'anacat':signal_cats, 'systematic':'nominal'}, mass[i])
        hep.histplot(RSGluonhists2, ax=bx, color=Color[i], lw=3, label='Mass ['+mass[i]+' GeV]', yerr=False)
        yMax2.append(np.max(RSGluonhists2.values()))
        
    # plt.yscale('log')
    # plt.ylim(0, 100) 
    plt.xlim(800, 8000)
    plt.legend(loc='center left', bbox_to_anchor=(1,0.5))
    ax.text(6500, np.max(yMax1)*0.85, cat+'\nAPV', fontsize=25, fontweight='bold')
    bx.text(6500, np.max(yMax2)*0.85, cat+'\nnoAPV', fontsize=25, fontweight='bold')
    # plt.savefig("apv_"+anacats[icat] +".png")
    